<a href="https://colab.research.google.com/github/bisenaditya21/Google-AI-Training-Programs-III/blob/main/GoogleAIInternship3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PREDICTION (HORSE/HUMAN)

#Step 1: Upload kaggle.json
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"adityabisen21","key":"b1e68ca2b36d9c98dda79dcbdeb2e28f"}'}

In [ ]:
#step 2 Move kaggle.json(directory)
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#step3 Download Dataset
!kaggle datasets download -d sanikamal/horses-or-humans-dataset

Dataset URL: https://www.kaggle.com/datasets/sanikamal/horses-or-humans-dataset
License(s): other
100% 307M/307M [00:01<00:00, 195MB/s]



In [ ]:
#Step 4: Unzip
import zipfile
zip_ref = zipfile.ZipFile("horses-or-humans-dataset.zip")
zip_ref.extractall("/content/dataset")
zip_ref.close()

In [ ]:
#Step 5: Import Library
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
#Step 6: Dataset Path
train_dir = "/content/dataset/train"
validation_dir = "/content/dataset/validation"

In [ ]:
#Step 7: Data Generator

train_dir = "/content/dataset/horse-or-human/train"
validation_dir = "/content/dataset/horse-or-human/validation"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

validation_generator = val_datagen.flow_from_directory(
    validation_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

Found 1027 images belonging to 2 classes.
Found 256 images belonging to 2 classes.


In [ ]:
#Step 8: Build CNN Model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
    tf.keras.layers.Dropout(0.5)
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Step 9: Comppile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#Step 10: Train Model
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 76s 2s/step - accuracy: 0.4937 - loss: 8.1167 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.4946 - loss: 8.1046 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.5180 - loss: 7.7292 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - accuracy: 0.4869 - loss: 8.2287 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.4937 - loss: 8.1183 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.4985 - loss: 8.0391 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.4820 - loss: 8.3051 - val_accuracy: 0.5000 - val_loss: 0.7086
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.5054 - loss: 7.9308 - val_accuracy: 0.5000 - val_loss:

In [ ]:
#Step 11: Evaluate Model
loss, accuracy = model.evaluate(validation_generator)
print("Loss:", loss)
print("Accuracy:", accuracy)

8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 458ms/step - accuracy: 0.5000 - loss: 0.7086
Loss: 0.7086341977119446
Accuracy: 0.5


In [ ]:
#Step 12: Save Model
model.save("horse_human_model.keras")

In [ ]:
#Step 13: Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("horse_human_model.tflite", "wb") as f:
  f.write(tflite_model)

Saved artifact at '/tmp/tmpuje7kej4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 150, 150, 3), dtype=tf.float32, name='keras_tensor_10')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138568891077392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891081424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891083152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891082384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891083536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891083344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891083920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568936124112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891084304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568891079696: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
#Step 14: Download TFLite Model
from google.colab import files
files.download("horse_human_model.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>